In [ ]:
from Code.NetworkExpansion import *
from Code.externalize_results import *
import matplotlib.pyplot as plt
import networkx as nx

# Expansions from primordial seed sets
write_outputs = False
max_generation = 80

# define chemical scope
reaction_scope = get_sides(kegg_reactions)
compound_scope = set.union(*reaction_scope.stack())
seed_scope = seed_candidates.loc[seed_candidates.index.isin(compound_scope)]
seed_groups = ['JM', 'JM|G17', 'JM|G24', 'G24', 'JM|G24|TW', 'JM|G24|G17|TW']
seed_sets = {i: seed_scope.query('Source.str.contains("'+i+'")').index for i in seed_groups}

# run network expansions
results = dict()
for name, seeds in seed_sets.items():
    results[name] = expand_from_seed_set(seeds=seeds, reactions=reaction_scope, max_gen=max_generation)
    results[name]['bipartite_edgelist'] = make_bipartite_edgelist(results[name]['edge_pool'])

# summarize results
def result_to_df(key: str, to_apply):
    return pd.concat({i: results[i][key].apply(to_apply) for i in results.keys()}, axis=1)
summaries = {'compound count': result_to_df(key='cps', to_apply=lambda x: x.sum()),
             'reaction count': result_to_df(key='rns', to_apply=lambda x: x.sum()),
             'chiral fraction': result_to_df(key='chiral_fraction', to_apply=lambda x: x)}

# plot results
order = summaries['compound count'].iloc[-1].sort_values(ascending=False).index
fig, ax = plt.subplots(len(summaries.keys()), 1, figsize=(4, 7), sharex=True, layout='compressed')
for ict, iname in enumerate(summaries.keys()):
    summaries[iname].loc[:,order].plot(ax=ax[ict], ylabel=iname, legend=False, grid=True, xlabel='Expansion step', cmap='coolwarm')
ax[2].legend(loc='center', bbox_to_anchor=(0.5, -0.5), title='', ncols=2)
ax[2].axhline(0.5, color='gray', linestyle='dotted', lw=0.7)
plt.savefig('Output/all_seeds_expansion_comparison_notmerged.pdf', dpi=300, edgecolor='w', facecolor='w')
plt.show()
plt.close()

# write output files
if write_outputs:
    write_edgelist_csvs(results=results)
